# Geometric perturbation, short run

Ten images, four conditions each: no shift, then horizontal shifts of 1 px, 10 px, and 100 px. That is 10 generations and 40 inversions. The score is still `pearsonr(|z|, |z_hat|)`.

The shifts are horizontal only. Content moves right, and pixels that enter the frame are black. A 100 px shift on a 512 px image leaves a wide black strip; that is the intended large perturbation.

Results go to `results/geometric_registration/quick_horizontal`, separate from the 100-sample runs. Re-running the experiment cell continues this directory. Use the `gaussian-rotation` kernel on a one-GPU HiPerGator notebook session, with the working directory set to the repository root.

At about 3 seconds per generation and 3 seconds per inversion, this run is roughly 3 to 4 minutes of GPU time after the model has loaded.

## 1. Configuration

In [ ]:
from pathlib import Path

NUM_SAMPLES = 10
SEED = 42
OUTPUT_DIR = Path("results/geometric_registration/quick_horizontal")

MODEL_ID = "runwayml/stable-diffusion-v1-5"
NUM_INFERENCE_STEPS = 50
DEVICE = "cuda"
DTYPE = "float16"
IMAGE_SIZE = 512
PROMPT = "a photograph of an astronaut riding a horse"
GUIDANCE_SCALE = 7.5

CONDITIONS = [
    ("baseline", 0.0),
    ("horizontal_translation", 1.0),
    ("horizontal_translation", 10.0),
    ("horizontal_translation", 100.0),
]

## 2. Run the experiment

In [ ]:
import importlib.util
import json
import sys
from types import SimpleNamespace

import pandas as pd
import torch

ROOT = Path.cwd().resolve()
script = ROOT / "scripts" / "run_geometric_registration_experiment.py"
if not script.is_file():
    raise SystemExit(
        f"Cannot find {script}. Open Jupyter with the working directory set to the repository root."
    )
sys.path.insert(0, str(ROOT))
spec = importlib.util.spec_from_file_location("geometric_registration_experiment", script)
geom = importlib.util.module_from_spec(spec)
spec.loader.exec_module(geom)

from experiments.generate_and_invert import (
    _git_commit,
    _load_inversion_pipeline,
    _release_pipeline,
    _recovered_latent,
)

if DEVICE == "cuda" and not torch.cuda.is_available():
    raise SystemExit("cuda was requested but this session has no GPU")

dtype = geom._dtype_from_name(DTYPE)
output_dir = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else ROOT / OUTPUT_DIR
output_dir.mkdir(parents=True, exist_ok=True)
example_dir = output_dir / "examples"
csv_path = output_dir / "correlations.csv"

args = SimpleNamespace(
    num_samples=NUM_SAMPLES,
    seed=SEED,
    model_id=MODEL_ID,
    num_inference_steps=NUM_INFERENCE_STEPS,
    device=DEVICE,
    image_size=IMAGE_SIZE,
    prompt=PROMPT,
    guidance_scale=GUIDANCE_SCALE,
)

done_frame, done = geom._complete_samples(csv_path, len(CONDITIONS))
pending = [i for i in range(NUM_SAMPLES) if i not in done]
print(f"{len(done)} samples already complete, {len(pending)} remaining", flush=True)

if pending:
    generated, sample_seeds = geom._generate_samples(args, pending, dtype)
    inverse = _load_inversion_pipeline(MODEL_ID, dtype, DEVICE)
    frame = done_frame
    try:
        for sample_id, _seed, z, image in generated:
            z_np = z.numpy()
            rows = []
            for kind, magnitude in CONDITIONS:
                transformed = geom.apply_geometric(image, kind, magnitude)
                if sample_id == 0:
                    example_dir.mkdir(parents=True, exist_ok=True)
                    label = "baseline" if kind == "baseline" else f"{kind}_{magnitude:g}"
                    transformed.save(example_dir / f"sample000_{label}.png")
                inverted = inverse.invert(
                    prompt=PROMPT,
                    image=transformed,
                    num_inference_steps=NUM_INFERENCE_STEPS,
                    guidance_scale=GUIDANCE_SCALE,
                    inpaint_strength=1.0,
                    num_reg_steps=0,
                )
                z_hat = _recovered_latent(inverted.latents, z.shape)
                z_hat = z_hat.detach().to(device="cpu", dtype=torch.float32).numpy()
                rows.append(
                    {
                        "sample_id": sample_id,
                        "transformation_type": kind,
                        "magnitude": magnitude,
                        "correlation": geom.magnitude_correlation(z_np, z_hat),
                    }
                )
            frame = pd.concat([frame, pd.DataFrame(rows)], ignore_index=True)
            frame.to_csv(csv_path, index=False)
            print(f"inverted sample {sample_id}", flush=True)
    finally:
        _release_pipeline(inverse)
else:
    import numpy as np
    sample_seeds = np.random.default_rng(SEED).integers(0, np.iinfo(np.int32).max, size=NUM_SAMPLES)

metadata = {
    "seed": SEED,
    "num_samples": NUM_SAMPLES,
    "model_id": MODEL_ID,
    "image_size": IMAGE_SIZE,
    "dtype": DTYPE,
    "device": DEVICE,
    "prompt": PROMPT,
    "guidance_scale": GUIDANCE_SCALE,
    "num_inference_steps": NUM_INFERENCE_STEPS,
    "padding": "constant black RGB (0, 0, 0)",
    "translation_direction": "positive horizontal moves content right",
    "correlation": "pearsonr(|z|, |z_hat|)",
    "conditions": [{"transformation_type": k, "magnitude": m} for k, m in CONDITIONS],
    "sample_seeds": [int(s) for s in sample_seeds],
    "git_commit": _git_commit(),
}
(output_dir / "metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
print(f"Results: {csv_path}")

## 3. Summary

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

RUN_DIR = OUTPUT_DIR if OUTPUT_DIR.is_absolute() else ROOT / OUTPUT_DIR
raw = pd.read_csv(RUN_DIR / "correlations.csv")
summary = (
    raw.groupby("magnitude", sort=True)["correlation"]
    .agg(mean="mean", std="std", median="median", n="count")
    .reset_index()
)
summary.insert(0, "transformation", summary["magnitude"].map(lambda m: "none" if m == 0 else f"horizontal {m:g} px"))
display(summary.round(4))

## 4. Correlation versus shift

Error bars are ±1 standard error of the mean. Magnitude 0 is the untransformed image.

In [ ]:
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})
summary = summary.sort_values("magnitude")
sem = summary["std"] / np.sqrt(summary["n"])
x = np.arange(len(summary))
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.errorbar(x, summary["mean"], yerr=sem, fmt="o-", capsize=3)
ax.set_xticks(x, [f"{m:g}" for m in summary.magnitude])
ax.set(xlabel="horizontal shift (px)", ylabel="correlation of |z| and |z_hat|", title="Magnitude correlation")
fig.tight_layout()
fig.savefig(RUN_DIR / "correlation_vs_shift.png", bbox_inches="tight")
plt.show()

## 5. Distribution across the 10 samples

In [ ]:
magnitudes = sorted(raw.magnitude.unique())
groups = [raw.loc[raw.magnitude == m, "correlation"].to_numpy() for m in magnitudes]
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.violinplot(groups, showmedians=True)
ax.set_xticks(range(1, len(magnitudes) + 1), [f"{m:g}" for m in magnitudes])
ax.set(xlabel="horizontal shift (px)", ylabel="correlation")
fig.tight_layout()
fig.savefig(RUN_DIR / "correlation_violins.png", bbox_inches="tight")
plt.show()

## 6. Matched comparison with no shift

`delta = correlation(shift) − correlation(no shift)` on the same image. Negative means the shifted image correlates less than the untransformed inversion.

In [ ]:
base = raw.loc[raw.magnitude == 0, ["sample_id", "correlation"]].rename(columns={"correlation": "baseline"})
matched = raw.loc[raw.magnitude != 0].merge(base, on="sample_id")
matched["delta"] = matched.correlation - matched.baseline
delta_summary = (
    matched.groupby("magnitude")["delta"]
    .agg(mean="mean", std="std", median="median", n="count")
    .reset_index()
)
display(delta_summary.round(4))

delta_summary = delta_summary.sort_values("magnitude")
sem = delta_summary["std"] / np.sqrt(delta_summary["n"])
x = np.arange(len(delta_summary))
fig, ax = plt.subplots(figsize=(5.5, 3.6))
ax.errorbar(x, delta_summary["mean"], yerr=sem, fmt="o-", capsize=3, color="crimson")
ax.axhline(0, color="k", lw=1)
ax.set_xticks(x, [f"{m:g}" for m in delta_summary.magnitude])
ax.set(xlabel="horizontal shift (px)", ylabel="delta correlation vs no shift")
fig.tight_layout()
fig.savefig(RUN_DIR / "delta_vs_shift.png", bbox_inches="tight")
plt.show()

## 7. Example shifts

Sample 0. Black pixels on the left are the fill from moving the image right.

In [ ]:
examples = [
    ("no shift", RUN_DIR / "examples" / "sample000_baseline.png"),
    ("1 px", RUN_DIR / "examples" / "sample000_horizontal_translation_1.png"),
    ("10 px", RUN_DIR / "examples" / "sample000_horizontal_translation_10.png"),
    ("100 px", RUN_DIR / "examples" / "sample000_horizontal_translation_100.png"),
]
fig, axes = plt.subplots(1, 4, figsize=(12, 3.2))
for ax, (title, path) in zip(axes, examples):
    ax.axis("off")
    ax.set_title(title, fontsize=10)
    if path.is_file():
        ax.imshow(plt.imread(path))
    else:
        ax.text(0.5, 0.5, "not saved yet", ha="center", va="center")
fig.tight_layout()
plt.show()

## 8. What this run shows

This printout only describes these 10 images. It does not assume the correlation falls as the shift grows.

In [ ]:
means = summary.sort_values("magnitude")
print("mean correlation by shift (px):")
for row in means.itertuples(index=False):
    print(f"  {row.magnitude:6.0f}  {row.mean:.4f}")
steps = np.diff(means["mean"].to_numpy())
print("non-increasing:" if len(steps) and np.all(steps <= 1e-12) else "not monotone")